In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import joblib
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

# 1. Define explicit absolute paths based on your Windows structure
BASE_DIR = r"C:\Users\SINCHANA\OneDrive\Desktop\customer-churn-prediction"
DATA_PATH = os.path.join(BASE_DIR, "data", "WA_Fn-UseC_-Telco-Customer-Churn.csv")
MODEL_PATH = os.path.join(BASE_DIR, "models", "model.pkl")
PREPROCESSOR_PATH = os.path.join(BASE_DIR, "models", "preprocessor.pkl")

print("--- Starting Clean Model Training Pipeline ---")

# 2. Read Dataset
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Missing data file at: {DATA_PATH}. Please make sure the CSV is there.")

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['customerID'], errors='ignore')

# Clean target & numeric features
df['TotalCharges'] = df['TotalCharges'].replace(' ', np.nan)
df['TotalCharges'] = df['TotalCharges'].astype(float).fillna(0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

X = df.drop(columns=['Churn'])
y = df['Churn']

# 3. Create Transformation Pipeline
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [col for col in X.columns if col not in num_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# Split and process data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# 4. Train a Fresh XGBoost Model
model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train_transformed, y_train)

# Quick validation check
preds = model.predict(X_test_transformed)
print(f"Success! Model Trained with Accuracy: {accuracy_score(y_test, preds) * 100:.2f}%")

# 5. Save Fresh Files (This overwrites any corrupted 0 KB files completely)
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

with open(MODEL_PATH, 'wb') as f:
    joblib.dump(model, f)
    
with open(PREPROCESSOR_PATH, 'wb') as f:
    joblib.dump(preprocessor, f)

print(f" Saved clean model to: {MODEL_PATH}")
print(f" Saved clean preprocessor to: {PREPROCESSOR_PATH}")